In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "126ac41d-e5e6-4702-b949-9d141a1b7042"
RESOURCE_GROUP = "rg-mlab1-ali886"
WORKSPACE = "mlw-lab1-ali886"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
                     RESOURCE_GROUP, WORKSPACE)
print("Connected:", ml_client.workspace_name)

In [ ]:
try:
    ml_client.online_endpoints.begin_delete(name="knn-maint-ali886").result()
    print("Old endpoint deleted")
except Exception as e:
    print("Delete message:", e)

In [ ]:
ENDPOINT_NAME = "knn-maint-ali886"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created:", ENDPOINT_NAME)

In [ ]:
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

In [ ]:
import json

sample = {"input_data": [[298.1, 308.6, 1551, 42.8, 0, 0]]}  # apne features ke hisaab se
with open("../deployment/sample.json", "w") as f:
    json.dump(sample, f)

result = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file="../deployment/sample.json",
)
print(result)

In [ ]:
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print(model.type, model.path)

import sklearn
print(sklearn.__version__)

In [ ]:
import os
from azure.ai.ml.entities import Environment

# 1) conda.yml banao
os.makedirs("../deployment", exist_ok=True)
open("../deployment/conda.yml", "w").write('''
name: knn-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - mlflow
      - cloudpickle
      - scikit-learn==1.7.2
      - numpy
      - pandas
      - azureml-inference-server-http
''')

# 2) environment
env = Environment(
    name="knn-mlflow-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.2-ubuntu20.04:latest",
    conda_file="../deployment/conda.yml",
)

# 3) model
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

# 4) deployment
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    environment=env,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

# 5) traffic
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

In [ ]:
import os
from azure.ai.ml.entities import Environment

os.makedirs("../deployment", exist_ok=True)
open("../deployment/conda.yml", "w").write('''
name: knn-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - mlflow
      - cloudpickle
      - scikit-learn==1.7.2
      - numpy
      - pandas
      - azureml-inference-server-http
''')

env = Environment(
    name="knn-mlflow-env-v2",
    image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cpu-inference:latest",
    conda_file="../deployment/conda.yml",
)

model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    environment=env,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

In [ ]:
print(ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=ENDPOINT_NAME, lines=300))

In [ ]:
import os
from azure.ai.ml.entities import Environment, CodeConfiguration

os.makedirs("../deployment", exist_ok=True)

open("../deployment/score.py", "w").write('''
import os, json
import mlflow.pyfunc
import pandas as pd

def init():
    global model
    base = os.getenv("AZUREML_MODEL_DIR")
    model_path = base
    for root, _, files in os.walk(base):
        if "MLmodel" in files:
            model_path = root
            break
    model = mlflow.pyfunc.load_model(model_path)

def run(raw_data):
    data = json.loads(raw_data)["input_data"]
    if isinstance(data, dict):
        df = pd.DataFrame(data["data"], columns=data.get("columns"))
    else:
        df = pd.DataFrame(data)
    return model.predict(df).tolist()
''')

open("../deployment/conda.yml", "w").write('''
name: knn-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - mlflow
      - cloudpickle
      - scikit-learn==1.7.2
      - numpy
      - pandas
      - azureml-inference-server-http
''')

env = Environment(
    name="knn-mlflow-env-v3",
    image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cpu-inference:latest",
    conda_file="../deployment/conda.yml",
)

model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    environment=env,
    code_configuration=CodeConfiguration(code="../deployment", scoring_script="score.py"),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

In [ ]:
import json

sample = {"input_data": [[298.1, 308.6, 1551, 42.8, 0, 0]]}  # apne 6 features ke order mein
with open("../deployment/sample.json", "w") as f:
    json.dump(sample, f)

result = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file="../deployment/sample.json",
)
print(result)

In [ ]:
logs = ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=ENDPOINT_NAME, lines=300)
print(logs[-3500:])

In [ ]:
import json

sample = {
    "input_data": {
        "columns": ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min", "type"],
        "data": [
            [298.1, 308.6, 1551.0, 42.8, 0.0, "M"],     # normal machine
            [300.0, 309.5, 1300.0, 65.0, 220.0, "L"]    # high torque + high wear (failure-like)
        ]
    }
}

with open("../deployment/sample.json", "w") as f:
    json.dump(sample, f)

result = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file="../deployment/sample.json",
)
print(result)

In [ ]:
ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
print("Endpoint deleted")